# **SQL Data Analysis — Project 3**

## **Step 0 — Import Libraries**

In [3]:
# sqlite3 is built into Python — no installation needed!
import sqlite3
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


## **Step 1 — Load Dataset & Create SQL Database**

In [5]:
# Load the Excel file
df = pd.read_excel('Dataset for Data Analytics.xlsx')

# Apply Project 1 cleaning
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

print("Dataset loaded!")
print("Shape:", df.shape)
df.head()

Dataset loaded!
Shape: (1200, 14)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [6]:
# Create an in-memory SQL database
conn = sqlite3.connect(':memory:')

# Load the dataframe into it as a table called 'orders'
df.to_sql('orders', conn, index=False, if_exists='replace')

print("SQL database created!")
print("Table 'orders' is ready to query.")

SQL database created!
Table 'orders' is ready to query.


## **Step 2 — Basic SELECT Query**

### **Q1 — SELECT: View First 10 Orders**

In [8]:
query = """
SELECT OrderID, Date, Product, Quantity, UnitPrice, TotalPrice
FROM orders
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
result

,OrderID,Date,Product,Quantity,UnitPrice,TotalPrice
0,ORD200000,2023-01-04,Monitor,5,570.62,2853.10
1,ORD200001,2024-08-23,Phone,2,151.35,302.70
2,ORD200002,2024-02-27,Tablet,5,550.68,2753.40
3,ORD200003,2023-10-15,Chair,1,273.19,273.19
4,ORD200004,2025-05-08,Printer,4,626.01,2504.04
5,ORD200005,2023-10-23,Phone,2,245.86,491.72
6,ORD200006,2025-06-17,Laptop,1,664.42,664.42
7,ORD200007,2023-05-12,Monitor,5,149.55,747.75
8,ORD200008,2025-04-02,Phone,2,134.28,268.56
9,ORD200009,2023-11-21,Desk,4,509.38,2037.52


## **Step 3 — WHERE (Filtering)**

### **Q2 — WHERE: High-Value Orders (TotalPrice > 2000)**

In [10]:
query = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC
"""

result = pd.read_sql_query(query, conn)
print(f"Orders above ₹2000: {len(result)}")
result

Orders above ₹2000: 180


,OrderID,Product,TotalPrice,OrderStatus
0,ORD200789,Tablet,3456.40,Delivered
1,ORD201122,Monitor,3390.95,Returned
2,ORD200632,Laptop,3390.80,Delivered
3,ORD200469,Chair,3384.90,Cancelled
4,ORD200328,Tablet,3370.20,Cancelled
...,...,...,...,...
175,ORD201035,Laptop,2024.44,Cancelled
176,ORD200181,Printer,2021.95,Shipped
177,ORD200915,Laptop,2011.96,Shipped
178,ORD200979,Monitor,2007.18,Delivered


### **Q3 — WHERE: Delivered Orders Only**

In [12]:
query = """
SELECT OrderID, Product, TotalPrice, ReferralSource
FROM orders
WHERE OrderStatus = 'Delivered'
ORDER BY TotalPrice DESC
"""

result = pd.read_sql_query(query, conn)
print(f"Total Delivered orders: {len(result)}")
result

Total Delivered orders: 231


,OrderID,Product,TotalPrice,ReferralSource
0,ORD200789,Tablet,3456.40,Email
1,ORD200632,Laptop,3390.80,Facebook
2,ORD201065,Printer,3334.00,Referral
3,ORD200361,Printer,3299.25,Facebook
4,ORD200837,Chair,3277.75,Email
...,...,...,...,...
226,ORD200351,Laptop,64.66,Instagram
227,ORD201020,Tablet,53.00,Instagram
228,ORD201044,Printer,38.49,Referral
229,ORD200926,Desk,26.95,Email


## **Step 4 — COUNT (Aggregation)**

### **Q4 — COUNT: Total Number of Orders**

In [13]:
query = """
SELECT COUNT(*) AS Total_Orders
FROM orders
"""

result = pd.read_sql_query(query, conn)
result

,Total_Orders
0,1200


## **Step 5 — GROUP BY + COUNT**

###  **Q5 — GROUP BY: Orders per Product**

In [14]:
query = """
SELECT Product,
       COUNT(*) AS Total_Orders
FROM orders
GROUP BY Product
ORDER BY Total_Orders DESC
"""

result = pd.read_sql_query(query, conn)
result

,Product,Total_Orders
0,Printer,181
1,Tablet,179
2,Chair,178
3,Laptop,173
4,Desk,170
5,Monitor,163
6,Phone,156


## **Step 6 — GROUP BY + SUM**

### **Q6 — SUM + GROUP BY: Total Revenue per Product**

In [15]:
query = """
SELECT Product,
       COUNT(*) AS Orders,
       ROUND(SUM(TotalPrice), 2) AS Total_Revenue
FROM orders
GROUP BY Product
ORDER BY Total_Revenue DESC
"""

result = pd.read_sql_query(query, conn)
result

,Product,Orders,Total_Revenue
0,Chair,178,195620.11
1,Printer,181,195612.61
2,Laptop,173,192126.56
3,Tablet,179,186568.95
4,Monitor,163,175651.41
5,Desk,170,167459.93
6,Phone,156,151722.39


## **Step 7 — GROUP BY + AVG**

### **Q7 — AVG + GROUP BY: Avg Order Value by Payment Method**

In [16]:
query = """
SELECT PaymentMethod,
       COUNT(*) AS Orders,
       ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value
FROM orders
GROUP BY PaymentMethod
ORDER BY Avg_Order_Value DESC
"""

result = pd.read_sql_query(query, conn)
result

,PaymentMethod,Orders,Avg_Order_Value
0,Credit Card,234,1127.55
1,Gift Card,230,1070.97
2,Cash,246,1056.04
3,Online,258,1017.22
4,Debit Card,232,1001.56


## **Step 8 — ORDER BY + Multiple Aggregations**

### **Q8 — COUNT + AVG + GROUP BY: Orders by Status**

In [18]:
query = """
SELECT OrderStatus,
       COUNT(*) AS Total_Orders,
       ROUND(AVG(TotalPrice), 2) AS Avg_Value
FROM orders
GROUP BY OrderStatus
ORDER BY Total_Orders DESC
"""

result = pd.read_sql_query(query, conn)
result

,OrderStatus,Total_Orders,Avg_Value
0,Cancelled,250,1105.58
1,Returned,247,984.93
2,Pending,237,1081.55
3,Shipped,235,1047.49
4,Delivered,231,1050.22


## **Step 9 — Full Aggregation by Channel**

### **Q9 — GROUP BY: Revenue by Referral Source**

In [20]:
query = """
SELECT ReferralSource,
       COUNT(*) AS Total_Orders,
       ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
       ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value
FROM orders
GROUP BY ReferralSource
ORDER BY Total_Revenue DESC
"""

result = pd.read_sql_query(query, conn)
result

,ReferralSource,Total_Orders,Total_Revenue,Avg_Order_Value
0,Instagram,259,275285.45,1062.88
1,Email,250,261808.55,1047.23
2,Google,241,250441.48,1039.18
3,Facebook,228,250410.90,1098.29
4,Referral,222,226815.58,1021.69


## **Step 10 — Top N with ORDER BY + LIMIT**

### **Q10 — ORDER BY + LIMIT: Top 5 Highest Value Orders**

In [21]:
query = """
SELECT OrderID, CustomerID, Product,
       TotalPrice, OrderStatus, ReferralSource
FROM orders
ORDER BY TotalPrice DESC
LIMIT 5
"""

result = pd.read_sql_query(query, conn)
result

,OrderID,CustomerID,Product,TotalPrice,OrderStatus,ReferralSource
0,ORD200789,C57276,Tablet,3456.40,Delivered,Email
1,ORD201122,C38840,Monitor,3390.95,Returned,Facebook
2,ORD200632,C67260,Laptop,3390.80,Delivered,Facebook
3,ORD200469,C13877,Chair,3384.90,Cancelled,Facebook
4,ORD200328,C18404,Tablet,3370.20,Cancelled,Google


## **Step 11 — WHERE with AND (Multiple Filters)**

### **Q11 — WHERE with AND: Laptop Orders via Instagram**

In [22]:
query = """
SELECT OrderID, Product, TotalPrice, OrderStatus, Date
FROM orders
WHERE Product = 'Laptop'
  AND ReferralSource = 'Instagram'
ORDER BY TotalPrice DESC
"""

result = pd.read_sql_query(query, conn)
print(f"Laptop + Instagram orders: {len(result)}")
result

Laptop + Instagram orders: 40


,OrderID,Product,TotalPrice,OrderStatus,Date
0,ORD200463,Laptop,3313.90,Shipped,2023-05-26
1,ORD200367,Laptop,3293.85,Pending,2024-04-25
2,ORD200492,Laptop,3032.60,Shipped,2023-10-25
3,ORD200633,Laptop,3008.60,Cancelled,2024-04-10
4,ORD201087,Laptop,2772.28,Shipped,2025-03-23
5,ORD200608,Laptop,2715.80,Cancelled,2024-07-24
6,ORD200823,Laptop,2612.52,Shipped,2024-10-07
7,ORD201179,Laptop,2592.75,Cancelled,2023-01-08
8,ORD200446,Laptop,2547.90,Delivered,2023-10-18
9,ORD200357,Laptop,2042.80,Delivered,2023-07-08


## **Step 12 — Full Business Summary**

### **Q12 — Business Summary: COUNT + SUM + AVG + MIN + MAX**

In [24]:
query = """
SELECT COUNT(*)                     AS Total_Orders,
       ROUND(SUM(TotalPrice), 2)    AS Total_Revenue,
       ROUND(AVG(TotalPrice), 2)    AS Avg_Order_Value,
       ROUND(MIN(TotalPrice), 2)    AS Min_Order,
       ROUND(MAX(TotalPrice), 2)    AS Max_Order
FROM orders
"""

result = pd.read_sql_query(query, conn)
result

,Total_Orders,Total_Revenue,Avg_Order_Value,Min_Order,Max_Order
0,1200,1264761.96,1053.97,11.39,3456.4


## **Step 13 — Close the Connection**

In [25]:
conn.close()
print("Database connection closed. All queries complete!")

Database connection closed. All queries complete!
